# 第 48 章：Capstone 公开发布索引与课程维护清单

这个 notebook 用一个极小的 public-release table，对比“只按公开价值排序”的 naive baseline 和“先检查许可边界、维护责任人、过期材料与公开索引”的课程发布维护 workflow。

In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_public_release_maintenance_demo.csv')


def load_items(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['public_value_score'] = float(row['public_value_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_items(DATA_PATH)
print(f'Loaded {len(rows)} release-maintenance items from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Audience:', ordered_counts(rows, 'audience'))
print('License status:', ordered_counts(rows, 'license_status'))
print('Maintenance owner:', ordered_counts(rows, 'maintenance_owner_status'))
print('Stale material:', ordered_counts(rows, 'stale_material_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['release_id']
    for row in rows
    if row['reference_route'] == 'release_ready'
)
top4_value = sorted(
    rows,
    key=lambda row: row['public_value_score'],
    reverse=True,
)[:4]
baseline_ready_hits = sum(
    row['reference_route'] == 'release_ready'
    for row in top4_value
)
baseline_not_ready_hits = sum(
    row['reference_route'] != 'release_ready'
    for row in top4_value
)
baseline_ready_precision = baseline_ready_hits / len(top4_value)
baseline_not_ready_intrusion = baseline_not_ready_hits / len(top4_value)

print('Reference release-ready items:', reference_ready_ids)
print('Top-4 by public value:')
for row in top4_value:
    print(
        f"  {row['release_id']}: value={row['public_value_score']:.2f}, "
        f"route={row['reference_route']}, area={row['release_area']}"
    )
print('Baseline release precision:', round(baseline_ready_precision, 3))
print('Baseline not-ready intrusion:', round(baseline_not_ready_intrusion, 3))


In [ ]:
def route_release_item(row):
    if row['license_status'] in {'unclear', 'restricted'}:
        return 'resolve_license_boundary', 'license, privacy, or public-sharing boundary is not clear enough'
    if row['maintenance_owner_status'] != 'assigned':
        return 'assign_maintenance_owner', 'release item has no clear maintenance owner yet'
    if row['stale_material_status'] in {'stale', 'needs_review'}:
        return 'refresh_stale_material', 'material is stale or needs a freshness audit before release'
    if row['public_index_status'] != 'complete':
        return 'complete_public_index', 'public index entry, version, or link is not complete enough'
    return 'release_ready', 'item has clear boundary, owner, current material, and complete index entry'


routed_rows = []
for row in rows:
    route, reason = route_release_item(row)
    routed = dict(row)
    routed['route'] = route
    routed['reason'] = reason
    routed_rows.append(routed)

print('Public release workflow routes:')
for row in routed_rows:
    print(
        f"  {row['release_id']}: route={row['route']}, "
        f"reference={row['reference_route']}, reason={row['reason']}"
    )


In [ ]:
ready_queue = [row for row in routed_rows if row['route'] == 'release_ready']
index_queue = [row for row in routed_rows if row['route'] == 'complete_public_index']
license_queue = [row for row in routed_rows if row['route'] == 'resolve_license_boundary']
owner_queue = [row for row in routed_rows if row['route'] == 'assign_maintenance_owner']
stale_queue = [row for row in routed_rows if row['route'] == 'refresh_stale_material']

workflow_accuracy = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
) / len(routed_rows)
ready_precision = sum(
    row['reference_route'] == 'release_ready'
    for row in ready_queue
) / max(len(ready_queue), 1)

print('Release-ready queue:', [row['release_id'] for row in ready_queue])
print('Complete-index queue:', [row['release_id'] for row in index_queue])
print('Resolve-license queue:', [row['release_id'] for row in license_queue])
print('Assign-owner queue:', [row['release_id'] for row in owner_queue])
print('Refresh-stale queue:', [row['release_id'] for row in stale_queue])
print('Workflow route accuracy:', round(workflow_accuracy, 3))
print('Structured release precision:', round(ready_precision, 3))


In [ ]:
def maintenance_steps(row):
    steps = []
    if row['license_status'] in {'unclear', 'restricted'}:
        steps.append('先确认许可、隐私和公开边界；必要时把材料改为 internal 或 hold。')
    if row['maintenance_owner_status'] != 'assigned':
        steps.append('指定维护 owner，并记录检查频率、失效处理方式和截止时间。')
    if row['stale_material_status'] in {'stale', 'needs_review'}:
        steps.append('刷新过期材料：重跑 notebook、更新截图、检查依赖和课程日期。')
    if row['public_index_status'] != 'complete':
        steps.append('补完整公开索引：标题、目标读者、链接、版本、最后检查日期。')
    return steps or ['可以进入 public release index；发布时保留版本号和下一次检查日期。']


for row in routed_rows:
    if row['route'] != 'release_ready':
        print(f"\n{row['release_id']} -> {row['route']} ({row['release_area']})")
        for step in maintenance_steps(row):
            print(' -', step)


In [ ]:
release_index_outline = [
    'Resource title: student-facing or instructor-facing name',
    'Audience: students, TAs, instructors, or public readers',
    'Location: repository path, web page, PDF, or notebook entry point',
    'Boundary: public, internal, restricted, or hold',
    'Owner: person or role responsible for maintenance',
    'Freshness: last checked date, next check date, and known caveats',
    'Verification: notebook smoke test, link check, or classroom trial evidence',
]

maintenance_assistant_prompt = '''你是我的 capstone 公开发布与课程维护助手。

任务：
1. 先阅读 public-release table，不要只按 public value 排序；
2. 先检查 license boundary；
3. 再检查 maintenance owner、stale material 和 public index；
4. 把每个资源 route 到 release_ready、complete_public_index、
   resolve_license_boundary、assign_maintenance_owner 或 refresh_stale_material；
5. 对每个非 ready 资源输出发布前维护 checklist。

输出格式：
- Release-ready resources
- Complete-index resources
- Resolve-license resources
- Assign-owner resources
- Refresh-stale resources
- Maintenance checklist by resource
'''

print('Public release index outline:')
for item in release_index_outline:
    print(' -', item)

print('\nAssistant prompt:')
print(maintenance_assistant_prompt)


In [ ]:
release_snapshot = {
    'dataset': DATA_PATH.name,
    'n_release_items': len(rows),
    'baseline_top4_release_precision': round(baseline_ready_precision, 3),
    'baseline_not_ready_intrusion': round(baseline_not_ready_intrusion, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'release_ready': [row['release_id'] for row in ready_queue],
    'complete_public_index': [row['release_id'] for row in index_queue],
    'resolve_license_boundary': [row['release_id'] for row in license_queue],
    'assign_maintenance_owner': [row['release_id'] for row in owner_queue],
    'refresh_stale_material': [row['release_id'] for row in stale_queue],
    'python_version': platform.python_version(),
}

print('Public release maintenance snapshot:')
for key, value in release_snapshot.items():
    print(f'  {key}: {value}')
